# 02 · Agents with LangChain and LangGraph

**Session 5 — Agent Architecture and Ecosystem Tools · MBRF Data Analytics**

In notebook 01 we wrote the agent loop by hand. Here we rebuild the same agent with **LangChain** and look at what a framework adds:

1. Tools from plain Python functions.
2. `create_agent`: the loop, already written.
3. Switching model provider in one line.
4. Middleware: limits and **human approval** before sensitive tools.
5. Memory across turns.
6. **LangGraph**: when the steps should be fixed by code (a *workflow*) rather than chosen by the model (an *agent*).

> Many tutorials online use the pre-1.0 LangChain API (`AgentExecutor`, `initialize_agent`, chains). This notebook uses LangChain 1.x.

## 0. Setup

In [ ]:
%pip install -q "langchain>=1.5,<2" "langchain-openai" "langchain-google-genai" "langgraph>=1.2,<2"

In [ ]:
import os
import sqlite3
import urllib.request
from pathlib import Path
from typing import Literal

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.tools import tool

PROVIDER = "openai"  # "openai" | "gemini"

MODELS = {"openai": "gpt-5.4-mini", "gemini": "gemini-3.5-flash-lite"}
KEY_NAMES = {"openai": "OPENAI_API_KEY", "gemini": "GEMINI_API_KEY"}


def get_api_key(name: str) -> str:
    """Colab secrets (key icon on the left) > environment variable or .env file > ask."""
    try:
        from dotenv import load_dotenv

        load_dotenv()
    except ImportError:
        pass
    try:
        from google.colab import userdata

        return userdata.get(name)
    except Exception:
        pass
    if os.environ.get(name):
        return os.environ[name]
    from getpass import getpass

    return getpass(f"{name}: ")


# "provider:model" picks the LangChain integration; the rest of the notebook does not depend on it.
LC_PROVIDERS = {"openai": "openai", "gemini": "google_genai"}
model = init_chat_model(f"{LC_PROVIDERS[PROVIDER]}:{MODELS[PROVIDER]}", api_key=get_api_key(KEY_NAMES[PROVIDER]), max_retries=5)

In [ ]:
# Synthetic bank database used by the tools: from the repo if cloned, otherwise downloaded (Colab).
DATA_URL = "https://raw.githubusercontent.com/albertotb/curso-agentes-ia/main/data/raw"
DB_PATH = Path("../../data/raw/bank.db")
if not DB_PATH.exists():
    DB_PATH = Path("bank.db")
    if not DB_PATH.exists():
        urllib.request.urlretrieve(f"{DATA_URL}/bank.db", DB_PATH)

db = sqlite3.connect(DB_PATH, check_same_thread=False)
db.row_factory = sqlite3.Row

## 1. Tools from Python functions

With `@tool`, the function **name** becomes the tool name, the **docstring** the description, and the **type hints** the JSON Schema. A `Literal[...]` becomes an `enum`: the model can only choose among those values.

In [ ]:
@tool(parse_docstring=True)
def get_capital_ratios(reference_date: str) -> dict:
    """Get Demo Bank's regulatory capital and liquidity ratios (CET1, Tier 1, total capital, leverage, LCR) at a quarter end.

    Use it for any question about the bank's own capital or liquidity. Ratios are already computed: do not recompute them.

    Args:
        reference_date: Quarter-end date in YYYY-MM-DD format, e.g. 2026-09-30.
    """
    row = db.execute("SELECT * FROM capital_position WHERE reference_date = ?", (reference_date,)).fetchone()
    if row is None:
        available = [r[0] for r in db.execute("SELECT reference_date FROM capital_position ORDER BY 1")]
        return {"error": f"No data for {reference_date}. Available quarter ends: {available}"}
    rwa = row["rwa_credit_eur_m"] + row["rwa_market_eur_m"] + row["rwa_operational_eur_m"]
    tier1 = row["cet1_capital_eur_m"] + row["at1_capital_eur_m"]
    return {
        "reference_date": reference_date,
        "total_rwa_eur_m": round(rwa, 1),
        "cet1_ratio_pct": round(100 * row["cet1_capital_eur_m"] / rwa, 2),
        "tier1_ratio_pct": round(100 * tier1 / rwa, 2),
        "total_capital_ratio_pct": round(100 * (tier1 + row["tier2_capital_eur_m"]) / rwa, 2),
        "leverage_ratio_pct": round(100 * tier1 / row["leverage_exposure_eur_m"], 2),
        "lcr_pct": round(100 * row["hqla_eur_m"] / row["net_outflows_30d_eur_m"], 1),
    }


@tool(parse_docstring=True)
def get_customer_loans(customer_id: str) -> dict:
    """Get a customer's profile (segment, internal rating 1=best..8=default) and their loans with outstanding amount, IFRS 9 stage and days past due.

    Args:
        customer_id: Customer ID, e.g. C-0013.
    """
    customer = db.execute("SELECT customer_id, name, segment, internal_rating FROM customers WHERE customer_id = ?", (customer_id,)).fetchone()
    if customer is None:
        return {"error": f"Customer {customer_id} not found. IDs look like C-0001."}
    loans = db.execute("SELECT loan_id, product, outstanding_eur, ifrs9_stage, days_past_due FROM loans WHERE customer_id = ?", (customer_id,)).fetchall()
    return {**dict(customer), "loans": [dict(r) for r in loans]}


@tool(parse_docstring=True)
def get_portfolio_summary(segment: Literal["retail", "sme", "corporate", "all"] = "all") -> list[dict]:
    """Outstanding loan amount and number of loans by IFRS 9 stage for a customer segment.

    Args:
        segment: Customer segment to summarise, or "all".
    """
    query = "SELECT l.ifrs9_stage, COUNT(*) AS n_loans, ROUND(SUM(l.outstanding_eur)) AS outstanding_eur FROM loans l JOIN customers c USING (customer_id)"
    params: tuple = ()
    if segment != "all":
        query += " WHERE c.segment = ?"
        params = (segment,)
    return [dict(r) for r in db.execute(query + " GROUP BY l.ifrs9_stage ORDER BY 1", params)]


print(get_portfolio_summary.name)
print(get_portfolio_summary.description)
print(get_portfolio_summary.args)  # the JSON Schema the model will see

## 2. `create_agent`: the loop, already written

Model + tools + system prompt. Underneath it runs exactly the loop from notebook 01.

In [ ]:
SYSTEM_PROMPT = "You are an assistant for Demo Bank's risk department. Answer only from tool results; if the tools cannot answer, say so. Be concise and quote figures with their reference date."
TOOLS = [get_capital_ratios, get_customer_loans, get_portfolio_summary]

agent = create_agent(model, tools=TOOLS, system_prompt=SYSTEM_PROMPT)

result = agent.invoke({"messages": [{"role": "user", "content": "How much of our SME portfolio is in stage 2 or 3, and how did CET1 change between June and September 2026?"}]})
for m in result["messages"]:
    m.pretty_print()

The whole trace is there: the user question, each tool call with its arguments, each tool result, and the final answer. That trace is what you would **log for audit**.

## 3. Switching provider

Because LangChain wraps every provider behind the same interface, the agent does not change. Only the model line does, for example:

```python
model = init_chat_model("openai:gpt-5.4-mini")
model = init_chat_model("anthropic:claude-haiku-5-5")        # pip install langchain-anthropic
model = init_chat_model("google_genai:gemini-3.5-flash")     # pip install langchain-google-genai
```

Useful for comparing quality and cost, or for falling back to another provider when one fails (see `ModelFallbackMiddleware`).

## 4. Middleware: limits and human approval

Middleware hooks into the loop: before or after each model call, around each tool call. LangChain ships several ready-made ones; two matter immediately in a bank:

- **Limits** on model or tool calls, so a confused agent cannot loop (and spend) forever.
- **Human in the loop**: pause before a sensitive tool runs, show the call to a person, and continue only if they approve (or edit / reject it).

Pausing requires saving the agent's state somewhere (a **checkpointer**) so it can resume later — possibly much later, from another process.

In [ ]:
from langchain.agents.middleware import HumanInTheLoopMiddleware, ModelCallLimitMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

REVIEW_QUEUE = []


@tool(parse_docstring=True)
def get_cash_transactions(customer_id: str, min_amount_eur: float = 0) -> list[dict]:
    """List a customer's cash transactions in 2026, optionally above a minimum absolute amount.

    Args:
        customer_id: Customer ID, e.g. C-0046.
        min_amount_eur: Minimum absolute amount in EUR.
    """
    rows = db.execute(
        "SELECT tx_id, tx_date, amount_eur, counterparty_country, channel FROM transactions WHERE customer_id = ? AND tx_type = 'cash' AND ABS(amount_eur) >= ? ORDER BY tx_date",
        (customer_id, min_amount_eur),
    )
    return [dict(r) for r in rows]


@tool(parse_docstring=True)
def flag_customer_for_review(customer_id: str, reason: str) -> dict:
    """Open an AML review case for a customer. Use only after checking the customer's transactions.

    Args:
        customer_id: Customer ID, e.g. C-0046.
        reason: Short factual justification citing the transactions.
    """
    REVIEW_QUEUE.append({"customer_id": customer_id, "reason": reason})
    return {"status": "case opened", "customer_id": customer_id}


aml_agent = create_agent(
    model,
    tools=[get_cash_transactions, flag_customer_for_review],
    system_prompt="You are an AML analyst assistant at Demo Bank. Structuring = repeated cash operations just below 10,000 EUR.",
    middleware=[
        ModelCallLimitMiddleware(run_limit=10),
        HumanInTheLoopMiddleware(interrupt_on={"flag_customer_for_review": True}),  # reads run freely, the write waits for a human
    ],
    checkpointer=InMemorySaver(),
)

config = {"configurable": {"thread_id": "aml-case-1"}}
result = aml_agent.invoke({"messages": [{"role": "user", "content": "Check C-0046 for structuring and open a review case if needed."}]}, config, version="v2")

# The run stopped before flag_customer_for_review. What does it want to do?
for interrupt in result.interrupts:
    for request in interrupt.value["action_requests"]:
        print(request["name"], request["args"])

Now the human decides. Try `approve`, then rerun the two cells and try `reject` with a message.

In [ ]:
decision = {"type": "approve"}
# decision = {"type": "reject", "message": "Not enough evidence: check the counterparty countries first."}

result = aml_agent.invoke(Command(resume={"decisions": [decision]}), config, version="v2")
print(result.value["messages"][-1].text)
print(REVIEW_QUEUE)

## 5. Memory across turns

With a checkpointer, every call with the same `thread_id` continues the same conversation.

In [ ]:
chat_agent = create_agent(model, tools=TOOLS, system_prompt=SYSTEM_PROMPT, checkpointer=InMemorySaver())
config = {"configurable": {"thread_id": "analyst-42"}}

for question in ["What was our LCR at the end of September 2026?", "And three months earlier?", "Is that trend consistent with the CET1 trend?"]:
    result = chat_agent.invoke({"messages": [{"role": "user", "content": question}]}, config)
    print(f"> {question}\n{result['messages'][-1].text}\n")

## 6. LangGraph: workflows vs agents

`create_agent` lets **the model** decide the steps. Many banking processes should not work like that: the steps are known and must always happen, in order, and be auditable. That is a **workflow**: the steps are fixed by code, and the LLM is used only where language is needed.

LangGraph builds these as a graph: a shared **state**, **nodes** (plain functions) and **edges** (possibly conditional). `create_agent` itself is a small LangGraph graph.

Example: a quarterly capital check. Fetching and comparing numbers is deterministic; only the memo is written by the LLM, and only when needed.

In [ ]:
from typing import TypedDict

from langgraph.graph import END, START, StateGraph

# Illustrative CET1 requirement: Pillar 1 + Pillar 2 requirement (CET1 part) + capital conservation buffer + countercyclical buffer.
CET1_REQUIREMENT_PCT = 4.5 + 1.13 + 2.5 + 0.5
MANAGEMENT_BUFFER_BP = 300


class CapitalCheck(TypedDict, total=False):
    reference_date: str
    ratios: dict
    headroom_bp: float
    status: str
    memo: str


def fetch_ratios(state: CapitalCheck) -> CapitalCheck:
    return {"ratios": get_capital_ratios.invoke({"reference_date": state["reference_date"]})}


def assess(state: CapitalCheck) -> CapitalCheck:
    headroom = round(100 * (state["ratios"]["cet1_ratio_pct"] - CET1_REQUIREMENT_PCT))
    status = "red" if headroom < 0 else "amber" if headroom < MANAGEMENT_BUFFER_BP else "green"
    return {"headroom_bp": headroom, "status": status}


def draft_escalation_memo(state: CapitalCheck) -> CapitalCheck:
    prompt = (
        f"Write a 5-line escalation memo to the CRO. CET1 headroom over the requirement of {CET1_REQUIREMENT_PCT:.2f}% "
        f"is {state['headroom_bp']} bp (status {state['status']}), below the {MANAGEMENT_BUFFER_BP} bp management buffer. "
        f"Figures: {state['ratios']}. Use only these figures."
    )
    return {"memo": model.invoke(prompt).text}


def routine_note(state: CapitalCheck) -> CapitalCheck:
    return {"memo": f"{state['reference_date']}: CET1 {state['ratios']['cet1_ratio_pct']}%, headroom {state['headroom_bp']} bp. No action."}


graph = StateGraph(CapitalCheck)
graph.add_node("fetch_ratios", fetch_ratios)
graph.add_node("assess", assess)
graph.add_node("draft_escalation_memo", draft_escalation_memo)
graph.add_node("routine_note", routine_note)
graph.add_edge(START, "fetch_ratios")
graph.add_edge("fetch_ratios", "assess")
graph.add_conditional_edges("assess", lambda s: "routine_note" if s["status"] == "green" else "draft_escalation_memo", ["routine_note", "draft_escalation_memo"])
graph.add_edge("draft_escalation_memo", END)
graph.add_edge("routine_note", END)
capital_check = graph.compile()

for date in ["2026-06-30", "2026-09-30"]:
    out = capital_check.invoke({"reference_date": date})
    print(f"--- {date}: {out['status']} ---\n{out['memo']}\n")

In [ ]:
# The graph, drawn (Mermaid). Paste it into https://mermaid.live to render it.
print(capital_check.get_graph().draw_mermaid())

## When to use what

| | Write the loop yourself | `create_agent` | LangGraph workflow |
|---|---|---|---|
| Who decides the steps | the model | the model | **your code** |
| Good for | learning, small prototypes, full control | assistants with a handful of tools | regulated processes with fixed steps, approvals, audit |
| You get for free | nothing | loop, retries, limits, human approval, memory, provider switch | state, persistence, branching, pause/resume |

A framework does not make the agent smarter: it handles the plumbing. Quality still comes from **tools, data and prompts**.

Other frameworks you will hear about: OpenAI Agents SDK, Google ADK, Claude Agent SDK, PydanticAI, CrewAI, LlamaIndex. Same ideas, different trade-offs.

Next (Friday): give the agent a tool that **searches regulation** — RAG.